# Indian Equity Volatility Early-Warning Model

## Objective
Develop and evaluate machine-learning models that predict whether the Indian equity market will experience unusually high volatility over the next trading week, using historical market data.

## Approach
- Obtain historical NIFTY 50 market data.
- Engineer features using past market information only.
- Define a high-volatility prediction target.
- Compare a simple baseline with Logistic Regression, Random Forest, and XGBoost.
- Evaluate models using chronological validation and test data.

**Important:** The notebook reports exploratory results from chronological and walk-forward evaluation. Performance varies across market periods, and results from quarters with few high-volatility events should be interpreted cautiously. These findings do not establish future predictive performance or trading profitability.

> **Reproducibility note:** Run this notebook from top to bottom in a fresh Colab runtime. The target is defined once; the event threshold is learned from the training portion of the expanded-feature experiment; walk-forward metrics and repository outputs are generated from actual predictions. Test-period results have been inspected during development, so the test period is not an untouched final holdout.


In [ ]:

%pip -q install yfinance xgboost

import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

# Download historical NIFTY 50 index data
ticker = "^NSEI"

data = yf.download(
    ticker,
    period="10y",
    interval="1d",
    auto_adjust=True,
    progress=False
)

if data.empty:
    raise RuntimeError(
        "No market data was downloaded. Check the connection and retry; "
        "do not replace missing market data with synthetic data."
    )

# Handle possible multi-level column names returned by yfinance
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

data = data[["Open", "High", "Low", "Close", "Volume"]].copy()
data.index = pd.to_datetime(data.index)
data = data.sort_index()
data = data[~data.index.duplicated(keep="first")]

print("Dataset shape:", data.shape)
print("First date:", data.index.min().date())
print("Last date:", data.index.max().date())
print("\nFirst five rows:")
display(data.head())

print("\nMissing values:")
display(data.isna().sum())

print("\nData types:")
display(data.dtypes)



In [ ]:

print("Column names:")
print(data.columns.tolist())

print("\nIndex name:")
print(data.index.name)

print("\nFirst five rows:")
display(data.head())

print("\nSummary statistics:")
display(data.describe().round(2))

print("\nCheck for duplicate dates:")
print("Duplicate dates:", data.index.duplicated().sum())

print("\nCheck for invalid prices:")
print("Non-positive closing prices:", (data["Close"] <= 0).sum())



In [ ]:
# Build features using information available by the end of day t
# and define next-five-session realized volatility exactly once.

data["Daily_Return"] = np.log(data["Close"] / data["Close"].shift(1))

# At date t, use returns from t+1 through t+5.
daily_returns = data["Daily_Return"]
future_volatility = pd.Series(np.nan, index=data.index, dtype=float)
for i in range(len(data) - 5):
    next_five_returns = daily_returns.iloc[i + 1:i + 6]
    if next_five_returns.notna().all():
        future_volatility.iloc[i] = np.sqrt((next_five_returns ** 2).sum())

data["Future_5D_Volatility"] = future_volatility

data["Return_1D"] = data["Daily_Return"]
data["Return_5D"] = data["Close"].pct_change(5)
data["Return_20D"] = data["Close"].pct_change(20)
data["Volatility_5D"] = data["Daily_Return"].rolling(5).std()
data["Volatility_20D"] = data["Daily_Return"].rolling(20).std()
data["Intraday_Range"] = (data["High"] - data["Low"]) / data["Close"]
data["Open_Close_Return"] = (data["Close"] - data["Open"]) / data["Open"]
data["Volume_Change"] = data["Volume"].pct_change()

original_features = [
    "Return_1D", "Return_5D", "Return_20D",
    "Volatility_5D", "Volatility_20D", "Intraday_Range",
    "Open_Close_Return", "Volume_Change"
]

print("Target and original features created.")
print("Target rows available:", int(data["Future_5D_Volatility"].notna().sum()))
print("Feature columns:", original_features)
print("First usable rows:")
display(data[original_features + ["Future_5D_Volatility"]].dropna().head())


In [ ]:

# Additional features using information available up to day t

# Magnitude of the latest daily return
data["Abs_Return_1D"] = data["Daily_Return"].abs()

# Longer-term historical volatility
data["Volatility_10D"] = data["Daily_Return"].rolling(10).std()
data["Volatility_60D"] = data["Daily_Return"].rolling(60).std()

# Recent volatility relative to the 20-day level
data["Volatility_Ratio"] = (
    data["Volatility_5D"] /
    data["Volatility_20D"].replace(0, np.nan)
)

# Average absolute return over the past five sessions
data["Mean_Abs_Return_5D"] = (
    data["Daily_Return"].abs().rolling(5).mean()
)

# Downside risk: volatility of negative returns over 20 sessions
negative_returns = data["Daily_Return"].clip(upper=0)

data["Downside_Volatility_20D"] = (
    negative_returns.pow(2).rolling(20).mean().pow(0.5)
)

# Average intraday trading range over five sessions
data["Mean_Intraday_Range_5D"] = (
    data["Intraday_Range"].rolling(5).mean()
)

# Expanded feature list
expanded_features = [
    "Return_1D",
    "Return_5D",
    "Return_20D",
    "Volatility_5D",
    "Volatility_20D",
    "Intraday_Range",
    "Open_Close_Return",
    "Volume_Change",
    "Abs_Return_1D",
    "Volatility_10D",
    "Volatility_60D",
    "Volatility_Ratio",
    "Mean_Abs_Return_5D",
    "Downside_Volatility_20D",
    "Mean_Intraday_Range_5D"
]

print("Number of features:", len(expanded_features))
display(data[expanded_features].tail().round(5))


In [ ]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

# Build the expanded-feature modelling dataset. All features use information
# available by the end of the prediction date.
model_data_expanded = data.dropna(
    subset=expanded_features + ["Future_5D_Volatility"]
).copy()

# Chronological 70% / 15% / 15% split, with a five-row buffer at boundaries.
n = len(model_data_expanded)
train_end = int(n * 0.70)
val_end = int(n * 0.85)
gap = 5

train_exp = model_data_expanded.iloc[:train_end-gap].copy()
val_exp = model_data_expanded.iloc[train_end:val_end-gap].copy()
test_exp = model_data_expanded.iloc[val_end:].copy()

# Learn the event threshold from this experiment's training period only.
threshold = train_exp["Future_5D_Volatility"].quantile(0.80)
for split in (train_exp, val_exp, test_exp):
    split["High_Volatility"] = (
        split["Future_5D_Volatility"] > threshold
    ).astype(int)

# Replace infinities and cap Volume_Change using training-only quantiles.
for split in (train_exp, val_exp, test_exp):
    split[expanded_features] = split[expanded_features].replace(
        [np.inf, -np.inf], np.nan
    )
lower_v = train_exp["Volume_Change"].quantile(0.01)
upper_v = train_exp["Volume_Change"].quantile(0.99)
for split in (train_exp, val_exp, test_exp):
    split["Volume_Change"] = split["Volume_Change"].clip(
        lower=lower_v, upper=upper_v
    )

models_to_compare = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced", max_iter=2000, random_state=42
        ))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300, min_samples_leaf=10,
            class_weight="balanced", random_state=42, n_jobs=-1
        ))
    ]),
    "XGBoost": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(
            n_estimators=300, max_depth=3, learning_rate=0.03,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="logloss", random_state=42, n_jobs=-1
        ))
    ])
}

comparison_rows = []
for feature_label, feature_set in [
    ("Original 8 features", original_features),
    ("Expanded 15 features", expanded_features)
]:
    Xtr, ytr = train_exp[feature_set], train_exp["High_Volatility"]
    Xv, yv = val_exp[feature_set], val_exp["High_Volatility"]
    for model_name, base_model in models_to_compare.items():
        model = clone(base_model)
        model.fit(Xtr, ytr)
        probs = model.predict_proba(Xv)[:, 1]
        comparison_rows.append({
            "Feature_Set": feature_label,
            "Model": model_name,
            "ROC_AUC": roc_auc_score(yv, probs),
            "Average_Precision": average_precision_score(yv, probs)
        })

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    ["Feature_Set", "ROC_AUC", "Average_Precision"],
    ascending=[True, False, False]
).reset_index(drop=True)

print("Training-derived threshold:", round(threshold, 6))
for name, split in [("Train", train_exp), ("Validation", val_exp), ("Test", test_exp)]:
    print(f"{name}: {split.index.min().date()} to {split.index.max().date()} | "
          f"{len(split)} rows | event rate {split['High_Volatility'].mean():.1%}")
display(comparison_df.round(4))


In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# Use only the original 8 features
original_features = [
    "Return_1D", "Return_5D", "Return_20D",
    "Volatility_5D", "Volatility_20D",
    "Intraday_Range", "Open_Close_Return", "Volume_Change"
]

X_train = train_exp[original_features].replace(
    [np.inf, -np.inf], np.nan
)
y_train = train_exp["High_Volatility"]

X_val = val_exp[original_features].replace(
    [np.inf, -np.inf], np.nan
)
y_val = val_exp["High_Volatility"]

# Logistic Regression
logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)
logistic_scores = logistic_model.predict_proba(X_val)[:, 1]

# Baseline: recent 20-day volatility as the risk score
volatility_scores = val_exp["Volatility_20D"].to_numpy()

results = pd.DataFrame([
    {
        "Method": "Recent 20-day volatility",
        "ROC_AUC": roc_auc_score(y_val, volatility_scores),
        "Average_Precision": average_precision_score(
            y_val, volatility_scores
        )
    },
    {
        "Method": "Logistic Regression",
        "ROC_AUC": roc_auc_score(y_val, logistic_scores),
        "Average_Precision": average_precision_score(
            y_val, logistic_scores
        )
    }
])

display(results.round(4))


In [ ]:

from sklearn.metrics import precision_score, recall_score

# Compare risk scores on the same validation dates
y_true = val_exp["High_Volatility"].to_numpy()

scores = {
    "Recent 20-day volatility":
        val_exp["Volatility_20D"].to_numpy(),
    "Logistic Regression":
        logistic_model.predict_proba(
            val_exp[original_features]
            .replace([np.inf, -np.inf], np.nan)
        )[:, 1]
}

rows = []

for name, risk_scores in scores.items():
    # Flag the top 20% highest-risk days
    cutoff = np.quantile(risk_scores, 0.80)
    alerts = (risk_scores >= cutoff).astype(int)

    rows.append({
        "Method": name,
        "Alert_Rate": alerts.mean(),
        "Precision": precision_score(
            y_true, alerts, zero_division=0
        ),
        "Recall": recall_score(
            y_true, alerts, zero_division=0
        ),
        "Actual_Event_Rate": y_true.mean()
    })

display(pd.DataFrame(rows).round(4))


In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

original_features = [
    "Return_1D", "Return_5D", "Return_20D",
    "Volatility_5D", "Volatility_20D",
    "Intraday_Range", "Open_Close_Return", "Volume_Change"
]

logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ))
])

logistic_model.fit(
    train_exp[original_features],
    train_exp["High_Volatility"]
)

print("Model refitted successfully.")


In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

original_features = [
    "Return_1D", "Return_5D", "Return_20D",
    "Volatility_5D", "Volatility_20D",
    "Intraday_Range", "Open_Close_Return", "Volume_Change"
]

logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ))
])

logistic_model.fit(
    train_exp[original_features],
    train_exp["High_Volatility"]
)

print("Model refitted successfully.")


In [ ]:

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score
)

# Prepare untouched test data
X_test = test_exp[original_features].replace(
    [np.inf, -np.inf], np.nan
)
y_test = test_exp["High_Volatility"].to_numpy()

# Predictions from the model selected using validation data
test_scores = logistic_model.predict_proba(X_test)[:, 1]

# Baseline scores: recent 20-day volatility
baseline_scores = test_exp["Volatility_20D"].to_numpy()

def evaluate_top_20_percent(y_true, scores, method_name):
    cutoff = np.quantile(scores, 0.80)
    alerts = (scores >= cutoff).astype(int)

    return {
        "Method": method_name,
        "ROC_AUC": roc_auc_score(y_true, scores),
        "Average_Precision": average_precision_score(
            y_true, scores
        ),
        "Alert_Rate": alerts.mean(),
        "Precision": precision_score(
            y_true, alerts, zero_division=0
        ),
        "Recall": recall_score(
            y_true, alerts, zero_division=0
        ),
        "Actual_Event_Rate": y_true.mean()
    }

test_results = pd.DataFrame([
    evaluate_top_20_percent(
        y_test, baseline_scores, "Recent 20-day volatility"
    ),
    evaluate_top_20_percent(
        y_test, test_scores, "Logistic Regression"
    )
])

display(test_results.round(4))


In [ ]:

import matplotlib.pyplot as plt
import numpy as np

from sklearn.metrics import (
    RocCurveDisplay,
    PrecisionRecallDisplay,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Use the final test predictions already calculated
y_true = y_test
lr_scores = test_scores
baseline_scores = baseline_scores

# Apply the same top-20% alert rule
cutoff = np.quantile(lr_scores, 0.80)
lr_alerts = (lr_scores >= cutoff).astype(int)

# 1. ROC curve
plt.figure(figsize=(7, 5))
RocCurveDisplay.from_predictions(
    y_true, baseline_scores,
    name="20-day volatility baseline"
)
RocCurveDisplay.from_predictions(
    y_true, lr_scores,
    name="Logistic Regression"
)
plt.title("ROC Curve — High-Volatility Early Warning")
plt.tight_layout()
plt.show()

# 2. Precision-recall curve
plt.figure(figsize=(7, 5))
PrecisionRecallDisplay.from_predictions(
    y_true, baseline_scores,
    name="20-day volatility baseline"
)
PrecisionRecallDisplay.from_predictions(
    y_true, lr_scores,
    name="Logistic Regression"
)
plt.title("Precision-Recall Curve — Test Set")
plt.tight_layout()
plt.show()

# 3. Confusion matrix at the top-20% alert threshold
cm = confusion_matrix(y_true, lr_alerts, labels=[0, 1])

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No high volatility", "High volatility"]
)
disp.plot(values_format="d")
plt.title("Logistic Regression — Top 20% Risk Alerts")
plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Use the test-set results
y_true = y_test
lr_scores = np.asarray(test_scores)
baseline_scores = np.asarray(test_exp["Volatility_20D"])

# 1. ROC Curve
fpr_lr, tpr_lr, _ = roc_curve(y_true, lr_scores)
fpr_base, tpr_base, _ = roc_curve(y_true, baseline_scores)

plt.figure(figsize=(7, 5))
plt.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC = {auc(fpr_lr, tpr_lr):.3f})")
plt.plot(fpr_base, tpr_base, label=f"20-Day Volatility (AUC = {auc(fpr_base, tpr_base):.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Test Set: ROC Curve")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 2. Precision–Recall Curve
precision_lr, recall_lr, _ = precision_recall_curve(y_true, lr_scores)
precision_base, recall_base, _ = precision_recall_curve(y_true, baseline_scores)

plt.figure(figsize=(7, 5))
plt.plot(recall_lr, precision_lr, label="Logistic Regression")
plt.plot(recall_base, precision_base, label="20-Day Volatility")
plt.axhline(np.mean(y_true), linestyle="--", label="Test event rate")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Test Set: Precision–Recall Curve")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 3. Confusion Matrix at a Top-20% Risk Alert Threshold
cutoff = np.quantile(lr_scores, 0.80)
lr_alerts = (lr_scores >= cutoff).astype(int)

cm = confusion_matrix(y_true, lr_alerts, labels=[0, 1])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Normal volatility", "High volatility"]
)

disp.plot(values_format="d")
plt.title("Test Set: Confusion Matrix (Top 20% Risk Alerts)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


In [ ]:

from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd
import numpy as np

# 1. Check each split
for name, split in [
    ("Train", train_exp),
    ("Validation", val_exp),
    ("Test", test_exp)
]:
    print(f"\n{name}")
    print("Rows:", len(split))
    print("Date range:", split.index.min().date(), "to", split.index.max().date())
    print("Positive labels:", int(split["High_Volatility"].sum()))
    print("Event rate:", round(split["High_Volatility"].mean(), 4))

# 2. Recalculate model scores directly from the fitted model
original_features = [
    "Return_1D", "Return_5D", "Return_20D",
    "Volatility_5D", "Volatility_20D",
    "Intraday_Range", "Open_Close_Return", "Volume_Change"
]

val_scores_check = logistic_model.predict_proba(
    val_exp[original_features]
)[:, 1]

test_scores_check = logistic_model.predict_proba(
    test_exp[original_features]
)[:, 1]

# 3. Recalculate validation and test metrics
for name, split, scores in [
    ("Validation", val_exp, val_scores_check),
    ("Test", test_exp, test_scores_check)
]:
    y = split["High_Volatility"].to_numpy()
    print(f"\n{name} metrics")
    print("ROC-AUC:", round(roc_auc_score(y, scores), 4))
    print("Average Precision:", round(average_precision_score(y, scores), 4))
    print("Score range:", round(scores.min(), 4), "to", round(scores.max(), 4))

# 4. Check alignment and leakage risks
print("\nIntegrity checks")
print("Test rows match score count:", len(test_exp) == len(test_scores_check))
print("Duplicate dates in test:", test_exp.index.duplicated().sum())
print("Target accidentally included as a feature:",
      "High_Volatility" in original_features or
      "Future_5D_Volatility" in original_features)
print("Feature list:", original_features)


In [ ]:

from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd
import numpy as np

# Use the same fitted model and same 8 features
feature_cols = [
    "Return_1D", "Return_5D", "Return_20D",
    "Volatility_5D", "Volatility_20D",
    "Intraday_Range", "Open_Close_Return", "Volume_Change"
]

results = []

for name, split in [("Validation", val_exp), ("Test", test_exp)]:
    y = split["High_Volatility"].to_numpy()

    lr_scores = logistic_model.predict_proba(
        split[feature_cols]
    )[:, 1]

    baseline = split["Volatility_20D"].to_numpy()

    results.append({
        "Period": name,
        "LR_AUC": roc_auc_score(y, lr_scores),
        "LR_AP": average_precision_score(y, lr_scores),
        "Volatility_Baseline_AUC": roc_auc_score(y, baseline),
        "Volatility_Baseline_AP": average_precision_score(y, baseline),
        "Correlation_Vol20D_Target": np.corrcoef(
            split["Volatility_20D"], split["Future_5D_Volatility"]
        )[0, 1]
    })

print(pd.DataFrame(results).round(4).to_string(index=False))

# Compare feature distributions between validation and test
print("\nFeature distribution comparison:")
comparison = pd.DataFrame({
    "Validation_Median": val_exp[feature_cols].median(),
    "Test_Median": test_exp[feature_cols].median(),
    "Validation_Mean": val_exp[feature_cols].mean(),
    "Test_Mean": test_exp[feature_cols].mean()
})
print(comparison.round(5))

# Check whether high-volatility labels match the stated threshold
print("\nTarget threshold check:")
print("Threshold:", threshold)
for name, split in [("Validation", val_exp), ("Test", test_exp)]:
    expected = (
        split["Future_5D_Volatility"] > threshold
    ).astype(int)
    mismatches = (expected != split["High_Volatility"]).sum()
    print(name, "label mismatches:", int(mismatches))


In [ ]:

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Combine validation and test in chronological order
combined = pd.concat([val_exp, test_exp]).sort_index().copy()

# 1. Compare recent volatility with future realized volatility
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

axes[0].plot(
    combined.index, combined["Volatility_20D"],
    label="Recent 20-day volatility", alpha=0.8
)
axes[0].plot(
    combined.index, combined["Future_5D_Volatility"],
    label="Next 5-day realized volatility", alpha=0.8
)
axes[0].axvline(val_exp.index.max(), linestyle="--",
                label="Validation/Test boundary")
axes[0].set_title("Recent vs Future Volatility")
axes[0].legend()
axes[0].grid(alpha=0.25)

# Monthly high-volatility event rate
monthly_rate = combined["High_Volatility"].resample("ME").mean()
axes[1].plot(monthly_rate.index, monthly_rate.values, marker="o")
axes[1].axhline(0.20, linestyle="--", label="Training event rate (~20%)")
axes[1].axvline(val_exp.index.max(), linestyle="--",
                label="Validation/Test boundary")
axes[1].set_title("Monthly High-Volatility Event Rate")
axes[1].set_ylabel("Share of high-volatility days")
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

# 2. Calculate correlations separately by period
for name, split in [
    ("Train", train_exp),
    ("Validation", val_exp),
    ("Test", test_exp)
]:
    corr = split[
        ["Volatility_20D", "Future_5D_Volatility"]
    ].corr().iloc[0, 1]
    print(f"{name} correlation: {corr:.4f}")


In [ ]:

from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd
import numpy as np

# Recalculate predictions in chronological order
test_check = test_exp.copy()
test_check["LR_Score"] = logistic_model.predict_proba(
    test_check[feature_cols]
)[:, 1]
test_check["Baseline_Score"] = test_check["Volatility_20D"]

# Evaluate by calendar quarter
test_check["Quarter"] = test_check.index.to_period("Q")

rows = []
for quarter, group in test_check.groupby("Quarter"):
    y = group["High_Volatility"].to_numpy()

    # AUC requires both positive and negative examples
    if len(np.unique(y)) < 2:
        lr_auc = np.nan
        base_auc = np.nan
    else:
        lr_auc = roc_auc_score(y, group["LR_Score"])
        base_auc = roc_auc_score(y, group["Baseline_Score"])

    rows.append({
        "Quarter": str(quarter),
        "Observations": len(group),
        "High_Volatility_Days": int(y.sum()),
        "Event_Rate": round(y.mean(), 3),
        "LR_AUC": round(lr_auc, 3) if not np.isnan(lr_auc) else np.nan,
        "Baseline_AUC": round(base_auc, 3) if not np.isnan(base_auc) else np.nan
    })

quarterly_results = pd.DataFrame(rows)
print(quarterly_results.to_string(index=False))


In [ ]:

import matplotlib.pyplot as plt
import pandas as pd

audit = test_exp.copy()

audit["LR_Score"] = logistic_model.predict_proba(
    audit[feature_cols]
)[:, 1]

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

axes[0].plot(
    audit.index, audit["Future_5D_Volatility"],
    label="Future 5-day volatility"
)
axes[0].axhline(
    threshold, linestyle="--",
    label="High-volatility threshold"
)
axes[0].set_title("Test Period: Future Volatility and Threshold")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].step(
    audit.index, audit["High_Volatility"],
    where="mid"
)
axes[1].set_title("Actual High-Volatility Labels")
axes[1].set_ylabel("Label (0 or 1)")
axes[1].grid(alpha=0.3)

axes[2].plot(
    audit.index, audit["LR_Score"],
    label="Logistic Regression risk score"
)
axes[2].set_title("Model Risk Scores Over Time")
axes[2].set_ylabel("Predicted score")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Highest future-volatility observations:")
print(
    audit.nlargest(10, "Future_5D_Volatility")[
        ["Future_5D_Volatility", "High_Volatility", "LR_Score"]
    ].round(4).to_string()
)


In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd
import numpy as np

features = [
    "Return_1D", "Return_5D", "Return_20D",
    "Volatility_5D", "Volatility_20D",
    "Intraday_Range", "Open_Close_Return", "Volume_Change"
]

# Combine the existing chronological splits.
# The volatility threshold remains fixed from the original training period.
all_data = pd.concat([train_exp, val_exp, test_exp]).sort_index()
all_data = all_data.loc[~all_data.index.duplicated(keep="first")].copy()

# Evaluate each quarter in the existing test period.
walk_forward_results = []

for quarter in test_exp.index.to_period("Q").unique():
    quarter_mask = test_exp.index.to_period("Q") == quarter
    evaluation = test_exp.loc[quarter_mask].copy()

    # Use only observations before this quarter.
    # Remove the last 5 past rows to reduce overlap between training
    # targets and the evaluation period.
    first_eval_date = evaluation.index.min()
    past = all_data.loc[all_data.index < first_eval_date].iloc[:-5].copy()

    if len(past) < 200 or evaluation["High_Volatility"].nunique() < 2:
        print(f"Skipping {quarter}: insufficient training data or only one target class.")
        continue

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        ))
    ])

    model.fit(past[features], past["High_Volatility"])
    scores = model.predict_proba(evaluation[features])[:, 1]
    y = evaluation["High_Volatility"].to_numpy()

    walk_forward_results.append({
        "Quarter": str(quarter),
        "Train_Rows": len(past),
        "Evaluation_Rows": len(evaluation),
        "Positive_Days": int(y.sum()),
        "Event_Rate": round(y.mean(), 3),
        "ROC_AUC": round(roc_auc_score(y, scores), 3),
        "Average_Precision": round(average_precision_score(y, scores), 3)
    })

walk_forward_table = pd.DataFrame(walk_forward_results)
print("\nWalk-forward results:")
print(walk_forward_table.to_string(index=False))


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier
import pandas as pd
import numpy as np

features = original_features
all_data = pd.concat([train_exp, val_exp, test_exp]).sort_index()
all_data = all_data.loc[~all_data.index.duplicated(keep="first")].copy()

models_for_walk_forward = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced", max_iter=2000, random_state=42
        ))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300, min_samples_leaf=10,
            class_weight="balanced", random_state=42, n_jobs=-1
        ))
    ]),
    "XGBoost": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(
            n_estimators=200, max_depth=2, learning_rate=0.03,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=5,
            eval_metric="logloss", random_state=42, n_jobs=-1
        ))
    ])
}

model_results = []
for quarter in test_exp.index.to_period("Q").unique():
    evaluation = test_exp.loc[test_exp.index.to_period("Q") == quarter].copy()
    first_eval_date = evaluation.index.min()
    # Only data dated before this quarter; remove five most recent rows because
    # their five-session target windows may overlap the evaluation period.
    past = all_data.loc[all_data.index < first_eval_date].iloc[:-5].copy()
    y_eval = evaluation["High_Volatility"].to_numpy()
    if len(past) < 200 or past["High_Volatility"].nunique() < 2:
        print(f"Skipping {quarter}: insufficient past data or training class diversity.")
        continue

    for model_name, base_model in models_for_walk_forward.items():
        model = clone(base_model)
        model.fit(past[features], past["High_Volatility"])
        scores = model.predict_proba(evaluation[features])[:, 1]
        auc_value = (roc_auc_score(y_eval, scores)
                     if np.unique(y_eval).size == 2 else np.nan)
        ap_value = average_precision_score(y_eval, scores)
        model_results.append({
            "Quarter": str(quarter),
            "Model": model_name,
            "Train_Rows": len(past),
            "Observations": len(evaluation),
            "Positive_Days": int(y_eval.sum()),
            "Event_Rate": float(y_eval.mean()),
            "ROC_AUC": auc_value,
            "Average_Precision": ap_value
        })

comparison = pd.DataFrame(model_results)
if comparison.empty:
    raise RuntimeError("No quarterly walk-forward evaluations were produced.")
print("Walk-forward results (AUC is undefined when a quarter has only one class):")
display(comparison.round(4))


In [ ]:
# Build final results directly from model-generated walk-forward outputs.
final_results = comparison.rename(columns={
    "Quarter": "Evaluation Quarter",
    "Observations": "Test Observations",
    "Positive_Days": "High-Volatility Days",
    "ROC_AUC": "ROC-AUC",
    "Average_Precision": "Average Precision"
}).sort_values(["Evaluation Quarter", "Model"]).reset_index(drop=True)

display(final_results.round(4))


##  Final Model Evaluation and Research Findings

### Research Objective
This project investigates whether historical price movements and volatility indicators can help identify periods of elevated volatility in the NIFTY 50 index over the next five trading sessions.

### Methodology
- **Data:** Historical daily NIFTY 50 market data.
- **Features:** Eight indicators derived from historical returns, rolling volatility, intraday price ranges, and volume changes.
- **Target:** An indicator identifying whether future five-session volatility exceeds the 80th percentile of the training-period target distribution.
- **Models:** Logistic Regression, Random Forest, and XGBoost.
- **Evaluation:** Chronological data splitting and expanding-window walk-forward evaluation.

### Main Findings
1. Model performance varied considerably across evaluation quarters.
2. No single model consistently outperformed the others across all evaluated periods.
3. Random Forest and XGBoost performed competitively in some quarters, while Logistic Regression performed well in others.
4. Results were particularly unstable in quarters with very few positive observations.

### Limitations
- The analysis uses one index and a limited set of engineered features.
- Market conditions and the frequency of high-volatility events change over time.
- Several quarterly evaluation periods contain few positive observations, making their performance estimates uncertain.
- The test-period results have been inspected during project development and should not be considered an untouched final holdout.
- The findings are exploratory and do not establish profitability or demonstrate that the model is ready for live trading.

### Conclusion
The analysis illustrates the challenges of forecasting elevated market volatility and the importance of chronological evaluation. Further research should examine additional market indicators, longer evaluation periods, and robustness across different market regimes.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Write all repository-facing outputs into the notebook working directory,
# matching the README links and the current repository layout.
summary = (
    final_results.groupby("Model")
    .agg(
        Evaluated_Quarters=("Evaluation Quarter", "nunique"),
        Mean_ROC_AUC=("ROC-AUC", "mean"),
        Mean_Average_Precision=("Average Precision", "mean")
    )
    .reset_index()
)

final_results.to_csv("walk_forward_results.csv", index=False)
summary.to_csv("model_performance_summary.csv", index=False)

auc_pivot = final_results.pivot(
    index="Evaluation Quarter", columns="Model", values="ROC-AUC"
)
ax = auc_pivot.plot(kind="bar", figsize=(10, 5), ylim=(0, 1))
ax.axhline(0.5, linestyle="--", label="Random-ranking reference")
ax.set_title("Walk-Forward ROC-AUC by Quarter")
ax.set_xlabel("Evaluation Quarter")
ax.set_ylabel("ROC-AUC")
ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("walk_forward_auc.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

ap_pivot = final_results.pivot(
    index="Evaluation Quarter", columns="Model", values="Average Precision"
)
ax = ap_pivot.plot(kind="bar", figsize=(10, 5), ylim=(0, 1))
ax.set_title("Walk-Forward Average Precision by Quarter")
ax.set_xlabel("Evaluation Quarter")
ax.set_ylabel("Average Precision")
ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("walk_forward_average_precision.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("MODEL PERFORMANCE SUMMARY (descriptive; quarters are not equally informative)")
display(summary.round(4))
print("\nSaved files:")
for filename in [
    "walk_forward_results.csv", "model_performance_summary.csv",
    "walk_forward_auc.png", "walk_forward_average_precision.png"
]:
    print(filename)


In [ ]:
import shutil
import os

result_files = [
    "walk_forward_results.csv", "model_performance_summary.csv",
    "walk_forward_auc.png", "walk_forward_average_precision.png"
]
missing = [name for name in result_files if not os.path.exists(name)]
assert not missing, f"Run the results-export cell first. Missing: {missing}"

import zipfile
with zipfile.ZipFile("volatility_project_results.zip", "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for name in result_files:
        zf.write(name, arcname=name)
print("Created volatility_project_results.zip")


In [ ]:
readme = f"""# Indian Equity Volatility Early-Warning Model

## Overview
This project explores whether daily NIFTY 50 market data can help rank periods of elevated volatility over the next five trading sessions. It compares Logistic Regression, Random Forest, and XGBoost using chronological walk-forward evaluation.

## Research Question
Can historical returns, rolling volatility, intraday price ranges, and volume-related indicators help identify upcoming high-volatility periods?

## Data
- **Market:** NIFTY 50 index (`^NSEI`)
- **Source:** Yahoo Finance through `yfinance`
- **Frequency:** Daily
- **Window:** Approximately 10 years, depending on data availability when the notebook is rerun.

## Target
At date *t*, future five-session realized volatility is the square root of the sum of squared daily log returns for sessions *t+1* through *t+5*. A high-volatility event is defined using the 80th percentile of the training-period target distribution. The threshold is fixed for the main chronological split and reused for its later periods.

## Features
The main models use eight features available by the end of the prediction date: one-day, five-day, and twenty-day returns; five-day and twenty-day rolling volatility; intraday high-low range; open-to-close return; and volume change. An expanded 15-feature set is also explored during validation.

## Models and Evaluation
- Logistic Regression
- Random Forest
- XGBoost

The notebook uses chronological splits with a five-observation boundary gap and an expanding-window quarterly evaluation. ROC-AUC is undefined for quarters with only one target class; Average Precision can still be reported but is uncertain in small samples.

### Walk-forward summary

{summary.to_markdown(index=False, floatfmt='.4f')}

These are unweighted descriptive averages over the evaluated quarters. Quarter sizes and event rates differ, and some quarters have very few positive observations, so the averages are not conclusive evidence that one model is universally best.

## Repository Contents
- `Indian_Equity_Volatility_Model.ipynb` — feature engineering, modelling, evaluation, and export.
- `walk_forward_results.csv` — quarterly metrics generated by the notebook.
- `model_performance_summary.csv` — descriptive model summary generated by the notebook.
- `walk_forward_auc.png` — quarterly ROC-AUC chart.
- `walk_forward_average_precision.png` — quarterly Average Precision chart.

## Limitations
- The analysis covers one index and a limited set of engineered features.
- Market regimes and high-volatility event rates change over time.
- Small quarterly samples can make metrics unstable; ROC-AUC is omitted where only one class occurs.
- The test-period results were inspected during development and are not an untouched final holdout.
- Predictive ranking does not establish profitability or readiness for live trading.

## Reproducibility
Open the notebook in Google Colab and run cells from top to bottom. Dependencies are installed in the first code cell and market data are retrieved through `yfinance`. Since the source is updated over time, results may change on rerun. The notebook creates the CSV files and charts from the computed walk-forward results rather than manually entered metrics.

## Disclaimer
For educational and research purposes only. This project is not financial advice or a trading recommendation.
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)

print("README.md created from the current computed results.")


In [ ]:

from google.colab import files
import os

# Download the README
files.download("README.md")

# Download the results ZIP
files.download("volatility_project_results.zip")
